# 📓 Caderno de Exercícios — Aula 04: Análise Exploratória (EDA) e Negócio
**Capacitação de Python para Ciência de Dados — Case Campe Supply**

> 💡 **Hora de gerar valor para a Diretoria:** Com a base 100% limpa e auditada, nossa missão agora é responder às perguntas estratégicas de negócio da Campe Supply usando agrupamentos, agregações, tabelas dinâmicas e filtros.

In [ ]:
import pandas as pd
import numpy as np
from difflib import get_close_matches

# === Checkpoint Inicial (Base 100% Limpa das Aulas 01 a 03) ===
df = pd.read_excel('Campe_Supply_Corrompido.xlsx', sheet_name='Campe Supply')
df = df.drop(columns=['UNNAMED: 0', 'coluna_inutil_0', 'coluna_inutil_1'])
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('ã', 'a')
df = df.drop_duplicates().reset_index(drop=True)

categorias_oficiais = {
    'setor': ['Congelados', 'Latícinio', 'Bebidas', 'Carnes'],
    'produto': ['Queijo', 'Sorvete', 'Frango', 'Refrigerantes', 'Cervejas', 'Bovina', 'Leite', 'Lasanha', 'Pizza', 'Suína', 'Sucos', 'Coalhada'],
    'vendedor': ['Vanessa', 'Amaral', 'Leo', 'Daniele', 'Jonas', 'Evandro'],
    'regiao': ['Sudeste', 'Nordeste', 'Sul'],
    'estado': ['Rio de Janeiro', 'Minas Gerais', 'Bahia', 'Rio Grande do Norte', 'São Paulo', 'Sergipe', 'Rio Grande do Sul', 'Paraná', 'Santa Catarina'],
    'cliente': ['Atacarejo', 'Varejo', 'Atacado']
}
def corrigir_palavra(valor, lista_oficial):
    if pd.isna(valor): return np.nan
    match = get_close_matches(str(valor).strip(), lista_oficial, n=1, cutoff=0.6)
    return match[0] if match else np.nan

for col, lista_valida in categorias_oficiais.items():
    df[col] = df[col].apply(lambda x: corrigir_palavra(x, lista_valida))

mapa_produto_setor = {'Queijo': 'Latícinio', 'Leite': 'Latícinio', 'Coalhada': 'Latícinio', 'Sorvete': 'Congelados', 'Lasanha': 'Congelados', 'Pizza': 'Congelados', 'Frango': 'Carnes', 'Bovina': 'Carnes', 'Suína': 'Carnes', 'Refrigerantes': 'Bebidas', 'Cervejas': 'Bebidas', 'Sucos': 'Bebidas'}
mapa_estado_regiao = {'Rio de Janeiro': 'Sudeste', 'Minas Gerais': 'Sudeste', 'São Paulo': 'Sudeste', 'Bahia': 'Nordeste', 'Rio Grande do Norte': 'Nordeste', 'Sergipe': 'Nordeste', 'Rio Grande do Sul': 'Sul', 'Paraná': 'Sul', 'Santa Catarina': 'Sul'}
df['setor'] = df['setor'].fillna(df['produto'].map(mapa_produto_setor))
df['regiao'] = df['regiao'].fillna(df['estado'].map(mapa_estado_regiao))

for col in ['faturamento', 'margem_de_lucro', 'lucro']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.loc[df['lucro'] > df['faturamento'], 'lucro'] = df['faturamento'] * df['margem_de_lucro']
df.loc[(df['margem_de_lucro'] == 0) & (df['lucro'] > 0), 'margem_de_lucro'] = df['lucro'] / df['faturamento']
df.loc[(df['faturamento'] * df['margem_de_lucro'] - df['lucro']).abs() > 0.01, 'faturamento'] = df['lucro'] / df['margem_de_lucro']

df['lucro'] = df['lucro'].fillna(df['faturamento'] * df['margem_de_lucro']).round(2)
df['faturamento'] = df['faturamento'].fillna(df['lucro'] / df['margem_de_lucro']).round(2)
df['margem_de_lucro'] = df['margem_de_lucro'].fillna(df['lucro'] / df['faturamento']).round(2)
df = df.drop_duplicates().reset_index(drop=True)
df['mes'] = df['data_da_venda'].dt.month
print('Base limpa pronta para análise:', df.shape)

## 🎯 Missão 1: KPIs Gerais da Campe Supply
Calcule e imprima na tela os 4 grandes indicadores gerais da empresa:
1. **Faturamento Total** (`sum`)
2. **Lucro Líquido Total** (`sum`)
3. **Ticket Médio por Venda** (`mean` do faturamento)
4. **Margem de Lucro Média (%)** (`mean` da margem de lucro multiplicada por 100)

In [ ]:
# 1. Calcule e exiba os 4 KPIs gerais da empresa
# Seu código aqui:


## 🎯 Missão 2: Ranking de Vendedores (`groupby` + `agg`)
A diretoria comercial quer saber quem são os vendedores de melhor performance.
Crie uma tabela agrupada por `vendedor` mostrando:
* Quantidade de vendas realizadas (`count`)
* Faturamento Total (`sum`)
* Lucro Total (`sum`)
* Ticket Médio (`mean`)
* Margem Média (`mean`)

Ordene do maior para o menor **Lucro Total** (`sort_values`).
> 🧠 **Pergunta de Negócio:** O vendedor que faz mais vendas e gera mais faturamento também é o que tem o maior ticket médio?

In [ ]:
# 2. Monte o ranking completo de vendedores
# Seu código aqui:


## 🎯 Missão 3: Performance por Setor e Top Produtos
1. Faça um agrupamento por `setor` para descobrir qual setor fatura mais e qual dá mais lucro.
2. Faça um agrupamento duplo por `['setor', 'produto']` para listar os produtos mais lucrativos da empresa.

In [ ]:
# 3. Analise o desempenho por Setor e por Produto
# Seu código aqui:


## 🎯 Missão 4: Matriz Região vs. Canal de Cliente (`pivot_table`) e Filtros VIP (`.query()`)
1. Crie uma Tabela Dinâmica (`pd.pivot_table` ou `groupby().unstack()`) cruzando `regiao` nas linhas, `cliente` (*Atacado, Atacarejo, Varejo*) nas colunas e a soma de `faturamento` nos valores (dica: use `margins=True` para ver o total geral).
2. Filtre todas as **Vendas VIP do Sudeste** (vendas onde `regiao == 'Sudeste'` e `faturamento > 1500`) usando `.query()` ou máscara booleana.

In [ ]:
# 4. Construa a tabela dinâmica Região x Cliente e filtre as vendas VIP do Sudeste
# Seu código aqui:
